> ⚠️ **机器翻译提示**：本 notebook 由机器翻译生成，未经人工校对，如发现翻译错误敬请谅解（代码与计算结果和越南语原版完全一致，只有文字说明是机器翻译）。

# 第03单元 — 描述统计（240人班级数据）
第03单元专属 notebook —— 重现本单元课程中引用的所有数字：缺失数据、性别×教学方法交叉表、集中趋势与离散程度、按学校分组的箱线图。

SPSS/PSPP 对应操作：`FREQUENCIES`、`CROSSTABS`、`EXAMINE`。

In [ ]:
# 先运行这一格（Colab：点击 ▶）。无需安装任何库。
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """模拟数据集"240人班级" —— 第03-08单元统一使用 seed=2026，
    与仓库中的 lop_hoc_240.csv/.sav 完全一致（非真实数据，但100%可复现）。"""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["男", "女"], n)
    method = rng.choice(["传统教学", "项目式教学"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "项目式教学")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "项目式教学") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("已生成 df：", df.shape)

## 1. 缺失数据 —— 检查变量 `h2`
SPSS对应：`Analyze > Descriptive Statistics > Frequencies`，查看「Missing」一行。

🎯 **这一步用来回答什么问题？** 在计算任何统计量之前，都需要先知道有多少比例的数据缺失、是否存在某种规律（MCAR/MAR/MNAR）——因为处理缺失数据的方式（整案删除还是仅在需要时删除）会直接影响之后所有分析的可靠性。

In [ ]:
missing_h2 = df["h2"].isna().sum()
pct = missing_h2 / len(df) * 100
print(f"h2 缺失单元格数：{missing_h2} / {len(df)}（{pct:.2f}%）")

#### 📤 实际输出
`3 / 240（1.25%）`。✅ 与课程中引用的数字一致：缺失比例很小且无明显规律 → 涉及 `h2` 的相关分析中，采用逐对删除（pairwise）比删除全部3名学生更合理。

## 2. 性别×教学方法交叉表
SPSS对应：`Analyze > Descriptive Statistics > Crosstabs`，勾选 `Cells... > Row percentages`。

🎯 **这个方法用来回答什么问题？** 交叉表展示两个定类/定序变量的联合分布——用于凭肉眼初步判断两个变量之间是否可能存在关联，之后再进行正式检验（第07单元的卡方检验）。交叉表本身并不能说明这种关联是否具有统计显著性。

In [ ]:
ct_count = pd.crosstab(df["gender"], df["method"])
ct_pct = pd.crosstab(df["gender"], df["method"], normalize="index") * 100
display(ct_count)
display(ct_pct.round(1))

#### 📤 实际输出
男生（n=121）：项目式教学63人（52.1%），传统教学58人（47.9%）。女生（n=119）：项目式教学52人（43.7%），传统教学67人（56.3%）。✅ 与课程中的「交叉表陷阱」示例一致：原始计数63>52容易让人误以为差距很大，但按行百分比计算实际差距更小（52.1% vs 43.7%）。

## 3. 集中趋势与离散程度 —— 数学成绩
SPSS对应：`Analyze > Descriptive Statistics > Frequencies`，勾选 Mean/Median/Mode/Std deviation/Variance/Range。

In [ ]:
desc = df["math"].agg(["mean","median","std","var","min","max"])
mode_val = df["math"].mode().iloc[0]
print(desc.round(3))
print("众数：", mode_val)
print(f"全距 = {df['math'].max() - df['math'].min():.1f}")

#### 📤 实际输出
平均值≈47.91，标准差≈9.68 —— 这两个数字将作为第04单元置信区间计算的输入。

## 4. 说明「平均值相同，标准差差异巨大」（Cohen et al., pp.762–764）
三组人为数据，平均值均为6，但标准差差异很大 —— 说明为何只报告平均值永远不够。

In [ ]:
set1 = np.array([1, 2, 3, 4, 20])          # 有强离群值
set2 = np.array([4, 5, 6, 6, 9])           # 中等分散
set3 = np.array([5, 6, 6, 6, 7])           # 高度集中
for name, s in [("第1组（有离群值）", set1), ("第2组", set2), ("第3组（集中）", set3)]:
    print(f"{name}：平均值={s.mean():.2f}，标准差={s.std(ddof=1):.2f}")

#### 📤 实际输出
三组平均值均为6.00，但标准差分别约为7.91 / 1.87 / 0.71 —— 第(1)组的标准差是第(3)组的11倍，尽管平均值相同。✅ 与课程中的示例一致。

## 5. 按学校分组的数学成绩箱线图
SPSS对应：`Analyze > Descriptive Statistics > Explore`，Plots > Boxplot。

🎯 **这个方法用来回答什么问题？** 箱线图回答：分布是否对称、是否有离群值、各组之间如何错落对比——看形状比读原始数字表快得多。

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=df, x="school", y="math", order=["A","B","C"], ax=ax)
ax.set_title("按学校分组的数学成绩（n=240）")
plt.show()
df.groupby("school")["math"].agg(["count","mean","std"]).round(2)

#### 📤 实际输出
A校（n=80，平均值≈50.67）最高，C校（n=61，平均值≈45.34）最低 —— 同一数据在第06单元（方差分析 F(2,237)=5.87, p=0.003）中再次使用。